In [ ]:
import numpy as np
from typing import List, Dict, Any

class VectorDB:
  def __init__(self, normalize_for_cosine: bool = True) -> None:
    """Iniciar base de datos de vectores con búsqueda por cercanía. Permite almacenar vectores de dimensión fija N, cada uno asociado a un texto arbitrario, 
    y buscar los K vectores más cercanos a un vector de consulta usando métrica coseno o euclidiana. 
    
    Args:
      normalize_for_cosine (bool, optional): Si es True, los vectores se normalizan al agregarlos (norma L2 = 1) para acelerar la búsqueda coseno. Defaults to True.
    """
    self.vectors = []                                   # Lista de np.array normalizados si corresponde
    self.texts = []                                     # Texto asociado a cada vector
    self.normalize_for_cosine = normalize_for_cosine
    self._dim = None                                    # Dimensión de los vectores, se fija en el primer agregar

  def add(self, vector: List[float], text: str) -> None:
    """Agrega un vector y su texto asociado a la base de datos.

    Args:
      vector (List[float]): array de tamaño N
      text (str): texto descriptivo asociado al vector
    """
    vec = np.array(vector, dtype=float)
    # Validar dimensión
    if self._dim is None:
      self._dim = vec.shape[0]
    elif vec.shape[0] != self._dim:
      raise ValueError(f"Dimensión del vector debe ser {self._dim}, se recibió {vec.shape[0]}")

    # Normalizar si se requiere para búsqueda coseno rápida
    if self.normalize_for_cosine:
      norm = np.linalg.norm(vec)
      if norm > 0:
        vec = vec / norm
      # Si la norma es 0, se deja como está (vector cero)

    self.vectors.append(vec)
    self.texts.append(text)

  def search(self, query: List[float], k: int = 5, metric: str = "cosine") -> List[Dict[str, Any]]:
    """Busca los k vectores más cercanos al vector de consulta. 
    
    Args:
      query (List[float]): vector de consulta.
      k (int, optional): número de vecinos. Defaults to 5.
      metric (str, optional): "cosine" (distancia coseno) o "euclidean" (distancia euclidiana). Defaults to "cosine".

    Returns:
      List[Dict[str, float]]: Lista de diccionarios [{"vector": array, "text": str, "distance": float}] ordenada de menor a mayor distancia.
    """
    
    if not self.vectors:
      return []

    query_vec = np.array(query, dtype=float)
    if query_vec.shape[0] != self._dim:
      raise ValueError(f"Dimensión de la consulta debe ser {self._dim}")

    # Convertir la lista de vectores a una matriz numpy (M filas, N columnas)
    matrix = np.array(self.vectors)  # shape (M, N)

    # Calcular distancias según la métrica elegida
    if metric == "cosine":
      if self.normalize_for_cosine:
        # Normalizar el query igual que los vectores almacenados
        q_norm = np.linalg.norm(query_vec)
        if q_norm > 0:
          query_vec = query_vec / q_norm
        # Similitud coseno = producto punto (porque ambos están normalizados)
        similarities = np.dot(matrix, query_vec)      # shape (M,)
        distances = 1.0 - similarities                # distancia coseno
      else:
        # Coseno sin normalización previa
        dot_product = np.dot(matrix, query_vec)       # M
        matrix_norms = np.linalg.norm(matrix, axis=1)  # M
        q_norm = np.linalg.norm(query_vec)
        # Evitar división por cero; si algún vector es cero, similitud = 0, distancia = 1
        denom = matrix_norms * q_norm
        similarities = np.divide(dot_product, denom, out=np.zeros_like(dot_product), where=denom != 0)
        distances = 1.0 - similarities

    elif metric == "euclidean":
      diffs = matrix - query_vec   # broadcasting sobre M
      distances = np.linalg.norm(diffs, axis=1)

    else:
      raise ValueError("Métrica no soportada. Use 'cosine' o 'euclidean'")

    # Seleccionar los k índices con menor distancia
    effective_k = min(k, len(distances))
    # argpartition es eficiente, no ordena todo el arreglo
    candidate_idxs = np.argpartition(distances, effective_k - 1)[:effective_k]
    # Ordenar esos k índices por distancia ascendente
    sorted_idxs = candidate_idxs[np.argsort(distances[candidate_idxs])]

    results = []
    for i in sorted_idxs:
      results.append({
          "vector": self.vectors[i].tolist(), 
          "text": self.texts[i],
          "distance": float(distances[i])
      })
    return results

In [ ]:
# Crear base de datos con normalización para coseno (recomendado)
db = VectorDB(normalize_for_cosine=True)

# Agregar vectores con textos asociados (dimensión 3 en este ejemplo)
db.add([0.2, 0.5, 0.1], "gato")
db.add([0.3, 0.4, 0.8], "perro")
db.add([0.9, 0.1, 0.2], "coche")
db.add([0.2, 0.5, 0.1], "felino")   # Duplicado intencional

# Consulta de ejemplo
query = [0.25, 0.55, 0.15]

print("Búsqueda coseno (k=3):")
cosine_results = db.search(query, k=3, metric="cosine")
for r in cosine_results:
  print(f"  Text: {r['text']:12s} | Distance: {r['distance']:.4f} | Vector: {r['vector']}")

print("\nBúsqueda euclidiana (k=3):")
euclidean_results = db.search(query, k=3, metric="euclidean")
for r in euclidean_results:
  print(f"  Text: {r['text']:12s} | Distance: {r['distance']:.4f} | Vector: {r['vector']}")